<a href="https://colab.research.google.com/github/Dewdahara/CRMCM_CW1_w2054873/blob/main/CRMCM_CW1_w2054873.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6BUIS017W CRM & CM with Business Intelligence: Individual Coursework

## Task 1: Data Preparation and Feature Engineering

### Setup

 **Data snapshot update (10 November 2026):** Constituents and daily Close, Adjusted Close and Volume data are stored together in one frozen Excel workbook. Subsequent runs load this workbook and do not refresh Yahoo Finance prices. The benchmark (^GSPC) is included in the price worksheets. The current constituent list remains a point-in-time universe, not a historical membership archive; results are subject to survivorship and constituent-history bias.

In [1]:
# Install the extra packages: yfinance for the prices, lxml so pandas can read the Wikipedia table,
# and seaborn for the charts.
%pip install -q yfinance lxml seaborn

In [2]:
# Import the libraries and record the package versions used in this run.
import os, json, platform, warnings
from datetime import datetime, timezone
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display
import scipy
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

PACKAGE_VERSIONS = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "yfinance": yf.__version__,
    "matplotlib": matplotlib.__version__,
    "requests": requests.__version__,
    "seaborn": sns.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__
}
PACKAGE_VERSIONS

{'python': '3.13.18',
 'pandas': '2.2.3',
 'numpy': '2.4.6',
 'yfinance': '0.2.66',
 'matplotlib': '3.10.0',
 'requests': '2.32.4',
 'seaborn': '0.13.2',
 'scipy': '1.16.3',
 'scikit-learn': '1.6.1'}

In [3]:
# Settings for the study period and the data checks. The end date is exclusive, so the last day used
# is 31 Dec 2024.
START_DATE = "2021-01-01"
END_DATE = "2025-01-01"
BENCHMARK = "^GSPC"
BASE_WINDOW = 20
ACTIVITY_WINDOW = 60
HIGH_VOL_MULT = 2.0
MIN_AVAILABILITY = 0.80
MIN_WEEKS = 52

# Prices are never filled in, so dates before a listing stay missing.
FILL_PRICES = False
# Setting this to True downloads fresh constituents and prices and replaces the workbook.
# Once the workbook is saved, set it back to False for every later run.
CREATE_DATA_SNAPSHOT = False
SNAPSHOT_XLSX = "sp500_market_data_snapshot.xlsx"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

### 1.1 Ticker selection and data acquisition

In [4]:
# Load the saved Excel workbook with the constituents and price history. In a fresh Colab session,
# upload the workbook created earlier.
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

snapshot_loaded = False
if os.path.exists(SNAPSHOT_XLSX) and not CREATE_DATA_SNAPSHOT:
    snapshot_loaded = True
    print("Using existing Excel data snapshot:", SNAPSHOT_XLSX)
else:
    if CREATE_DATA_SNAPSHOT:
        print("CREATE_DATA_SNAPSHOT=True: creating a fresh snapshot; any existing workbook will be replaced.")
    try:
        from google.colab import files
        if not CREATE_DATA_SNAPSHOT:
            print("Upload the previously created sp500_market_data_snapshot.xlsx workbook.")
            uploaded = files.upload()
            for filename, content in uploaded.items():
                if filename.lower().endswith(".xlsx"):
                    with open(SNAPSHOT_XLSX, "wb") as f:
                        f.write(content)
                    snapshot_loaded = True
                    break
    except ImportError:
        pass

if snapshot_loaded:
    constituents = pd.read_excel(SNAPSHOT_XLSX, sheet_name="Constituents")
    metadata_df = pd.read_excel(SNAPSHOT_XLSX, sheet_name="Metadata")
    retrieval_info = dict(zip(metadata_df["key"].astype(str), metadata_df["value"].astype(str)))

    # Rename the constituent columns to the names used below. The workbook can have the original
    # Wikipedia headers or the cleaned ones, with small spacing or case differences.
    original_headers = list(constituents.columns)
    header_lookup = {str(c).strip().lower(): c for c in constituents.columns}
    aliases = {
        "symbol": ["symbol", "ticker", "ticker symbol", "yahoo ticker"],
        "security": ["security", "company", "company name", "name"],
        "gics sector": ["gics sector", "sector", "gics_sector"],
    }
    rename_map = {}
    for target, candidates in aliases.items():
        for candidate in candidates:
            if candidate in header_lookup:
                source_col = header_lookup[candidate]
                if source_col != target:
                    rename_map[source_col] = target
                break
    constituents = constituents.rename(columns=rename_map)
    print("Constituent worksheet columns found:", original_headers)
    print("Constituent columns standardised for analysis:", list(constituents.columns))
    print("Constituents and price data will be loaded from the same frozen workbook.")
else:
    if not CREATE_DATA_SNAPSHOT:
        raise FileNotFoundError(
            f"{SNAPSHOT_XLSX} was not found. Upload the previously created Excel workbook, "
            "or set CREATE_DATA_SNAPSHOT = True for its one-time creation."
        )
    # First-time creation only: get the constituent list here. Prices are downloaded in the next
    # cell.
    resp = requests.get(
        WIKI_URL,
        headers={"User-Agent": "Mozilla/5.0 (university coursework)"},
        timeout=30,
    )
    resp.raise_for_status()
    tables = pd.read_html(StringIO(resp.text))
    constituents = tables[0].copy()
    retrieval_info = {
        "constituents_source": WIKI_URL,
        "constituents_retrieved": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "universe_note": "Current S&P 500 constituents retrieved on the recorded date; not historical membership.",
        "price_source": f"Yahoo Finance via yfinance {yf.__version__}; auto_adjust=False",
    }
    print("Retrieved constituent snapshot for one-time Excel workbook creation.")

required_cols = {"symbol", "security", "gics sector"}
assert required_cols.issubset(constituents.columns), (
    f"Constituent data missing required columns: {required_cols - set(constituents.columns)}"
)
assert constituents["symbol"].notna().all(), "Constituent data contains missing ticker symbols."
assert constituents["symbol"].astype(str).is_unique, "Constituent data contains duplicate ticker symbols."

print("Constituents loaded:", len(constituents))
print("Constituent retrieval timestamp:", retrieval_info.get("constituents_retrieved", "Not available"))
print("Universe limitation:", retrieval_info.get("universe_note", "Point-in-time S&P 500 constituents; not historical membership."))

Upload the previously created sp500_market_data_snapshot.xlsx workbook.


Saving sp500_market_data_snapshot.xlsx to sp500_market_data_snapshot.xlsx
Constituent worksheet columns found: ['Ticker', 'Company', 'Sector']
Constituent columns standardised for analysis: ['symbol', 'security', 'gics sector']
Constituents and price data will be loaded from the same frozen workbook.
Constituents loaded: 503
Constituent retrieval timestamp: 2026-11-10 17:04 UTC
Universe limitation: Current S&P 500 constituents retrieved on the recorded date; not historical membership.


In [5]:
# Tidy up the constituents table. Tickers with a dot are changed to a dash for Yahoo Finance (BRK.B
# becomes BRK-B). The Excel file itself is not changed.
constituents["Ticker"] = constituents["symbol"].astype(str).str.replace(".", "-", regex=False)
constituents = constituents.rename(columns={"security": "Company", "gics sector": "Sector"})
constituents = constituents[["Ticker", "Company", "Sector"]].drop_duplicates("Ticker").reset_index(drop=True)

tickers = constituents["Ticker"].tolist()
print(f"{len(tickers)} tickers loaded from the frozen Excel snapshot")
print("No live Wikipedia scraping is performed in this cleaning/downstream step.")
display(constituents.head())

# Output:
# 503 tickers are loaded. The first five rows are MMM, AOS, ABT, ABBV and ACN with their company and
# sector.

503 tickers loaded from the frozen Excel snapshot
No live Wikipedia scraping is performed in this cleaning/downstream step.


,Ticker,Company,Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [6]:
# Load the saved price history from the workbook, or download it once when creating the workbook.
if snapshot_loaded:
    prices = {
        "close": pd.read_excel(SNAPSHOT_XLSX, sheet_name="Close", index_col=0, parse_dates=True),
        "adj_close": pd.read_excel(SNAPSHOT_XLSX, sheet_name="Adj_Close", index_col=0, parse_dates=True),
        "volume": pd.read_excel(SNAPSHOT_XLSX, sheet_name="Volume", index_col=0, parse_dates=True),
    }
    print("Historical prices loaded from the uploaded/saved Excel snapshot; Yahoo Finance was not queried.")
else:
    if not CREATE_DATA_SNAPSHOT:
        raise RuntimeError("No frozen Excel snapshot is loaded. Enable CREATE_DATA_SNAPSHOT only for one-time creation.")
    symbols = tickers + [BENCHMARK]
    frames = []
    for batch in np.array_split(symbols, 6):
        block = yf.download(
            [str(s) for s in batch], start=START_DATE, end=END_DATE,
            auto_adjust=False, actions=False, progress=True, threads=True,
        )
        frames.append(block)
    raw = pd.concat(frames, axis=1)
    prices = {"close": raw["Close"], "adj_close": raw["Adj Close"], "volume": raw["Volume"]}

    # Record where and when the data came from.
    retrieval_info["prices_downloaded"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    retrieval_info["price_source"] = f"Yahoo Finance via yfinance {yf.__version__}; auto_adjust=False; actions=False"
    retrieval_info["study_start_inclusive"] = START_DATE
    retrieval_info["study_end_exclusive"] = END_DATE
    retrieval_info["benchmark"] = BENCHMARK

    # Save everything into one Excel workbook, one sheet per table.
    metadata_df = pd.DataFrame({"key": list(retrieval_info.keys()), "value": [str(v) for v in retrieval_info.values()]})
    with pd.ExcelWriter(SNAPSHOT_XLSX, engine="openpyxl") as writer:
        constituents.to_excel(writer, sheet_name="Constituents", index=False)
        prices["close"].to_excel(writer, sheet_name="Close", index_label="Date")
        prices["adj_close"].to_excel(writer, sheet_name="Adj_Close", index_label="Date")
        prices["volume"].to_excel(writer, sheet_name="Volume", index_label="Date")
        metadata_df.to_excel(writer, sheet_name="Metadata", index=False)
    print("Created frozen Excel snapshot:", SNAPSHOT_XLSX)
    try:
        from google.colab import files
        files.download(SNAPSHOT_XLSX)
    except ImportError:
        print("Excel snapshot saved in the current working directory.")

# Keep only the constituent tickers and the benchmark.
keep_cols = set(tickers + [BENCHMARK])
for name, df in prices.items():
    df.columns = df.columns.astype(str)
    prices[name] = df.loc[:, [c for c in df.columns if c in keep_cols]]

missing_downloads = sorted(set(tickers) - set(prices["adj_close"].columns))
print("Tickers absent from downloaded price table:", missing_downloads)
print("Price download timestamp:", retrieval_info.get("prices_downloaded", "Not available in workbook"))

Historical prices loaded from the uploaded/saved Excel snapshot; Yahoo Finance was not queried.
Tickers absent from downloaded price table: []
Price download timestamp: 2026-11-10 17:06 UTC


In [7]:
# Check that the three tables cover the right dates and include the benchmark.
for name, df in prices.items():
    print(
        f"{name:10s} shape={df.shape}  first={pd.to_datetime(df.index.min()).date()}  last={pd.to_datetime(df.index.max()).date()}"
    )
print("Benchmark column present:", BENCHMARK in prices["adj_close"].columns)

# Output:
# Each table has 1005 trading days and 504 columns (503 tickers plus the S&P 500 benchmark), running
# from 2021-01-04 to 2024-12-31.
# The benchmark column is there and no 2025 dates are included.

close      shape=(1005, 504)  first=2021-01-04  last=2024-12-31
adj_close  shape=(1005, 504)  first=2021-01-04  last=2024-12-31
volume     shape=(1005, 504)  first=2021-01-04  last=2024-12-31
Benchmark column present: True
